# About this Continuous Assessment
|  |  |
|--|--|
|Assessment| 11|
|Delivery method | Canvas submission of this file after completion |
|Deadline | As specified on Canvas. |
|ILOs|Be able to apply hybrid force/motion controller in a contact-rich task.|

# Allowed packages 
Do not modify these. Do not add or remove packages.

In [10]:
# Helper functions and packages
%pip install numpy
%pip install matplotlib


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Allowed imports

<div class="alert alert-block alert-info">
Do not modify, add, or remove imports. Do not add imports anywhere else in this notebook.
</div>

In [11]:
# Helper functions and packages
import numpy as np

## Instructions
This assessment is made of 2 tasks. There are two points in total for this part of the assessment.   
Each cell is tagged for the assessment, so do not delete or otherwise tamper with the cells. The assignment is to add content to each of the Python cells as requested.

## Verify the spreadsheet

Your student ID will be used to assign you parameters. For each task, look into your row to see what you must implement.

# Control of a 7-DoF Manipulator for a contact-rich tooling task

The experimental platform consists of a 7-DoF robotic manipulator equipped with an end-effector tool for contact-rich interaction task (sliding) on a planar surface.  

A coordinate frame $o_cx_c\,y_c\,z_c$ is defined as the constraint frame, which is attached to the task space as shown in the figure. 

Assuming that the contact between the tool and the surface is rigid and frictionless. 

  <figure style="margin: 10px; width: 100%; text-align: left;">
    <img src="figures/figure_kinova.png" alt='A single-link robot arm.' style="max-width: 100%; height: 350px;"/>
    <figcaption>Figure 1 - A 7-DoF Kinova manipulator to conduct a fissure detection task on a planar surface.</figcaption>
  </figure>


---
## ✏️ Task 1 - Define the Task-Space Selection Matrices

The coordinates in constraint frame are defined as:

$$
\mathbf{X} =
\begin{bmatrix}
x & y & z & \theta_x & \theta_y & \theta_z
\end{bmatrix}^T
$$

where:

- $x, y$: translational motion along the surface plane,
- $z$: translational motion normal to the surface,
- $\theta_x, \theta_y, \theta_z$: rotational motion of the end-effector.

Complete the following code to define:

- $S_f$: the selection matrix for the constraint (force-controlled) subspace,
- $S_v$`: the selection matrix for the admissible motion subspace.

The matrices should correctly separate the task-space directions associated with motion control and force control.

In [12]:
# Code for Task 1
# For a sliding/fissure detection task on a planar surface:
# - x, y: motion along surface -> admissible (motion-controlled)
# - z: normal to surface -> constraint (force-controlled)
# - theta_x, theta_y, theta_z: rotations -> constraint (force-controlled)
# As per lecture notes (Section 9.1): position control in tangential plane (x,y),
# force control in normal direction (z) and rotational directions.

# Selection matrix for the constraint subspace (force-controlled: z, theta_x, theta_y, theta_z)
Sf = np.diag([0, 0, 1, 1, 1, 1])

# Selection matrix for the admissible motion subspace (motion-controlled: x, y)
Sv = np.diag([1, 1, 0, 0, 0, 0])


In [13]:
# Marking variables (do not modify)
ansT1_Sf = Sf
ansT1_Sv = Sv
# End of code for Task 1

---
## ✏️ Task 2 -  Compute the Joint-Space Force Control Input

In this task, you will implement the force-control component of the hybrid force/motion controller described in the lecture notes, using a proportional feedback controller with feedforward force compensation.

You are given the desired force $F_d$, measured force $F_e$, force-control gain $K_f$, Jacobian matrix $J$. Use the constraint selection matrix $S_f$ you defined in Task 1. 

Complete the code below to compute the joint-space force-control input $u_f$.

The implemented controller should include:
- a feedforward term to apply the desired contact force,
- and a feedback term to compensate for force tracking error.

In [14]:
# Define Force Control Parameters for Task 2
################################################################################
# TODO:   
# Enter the force control parameters below (Find them in the spreadsheet)
Kf = 0.30          # Proportional gain for force feedback control
Fd_z = 10          # Desired contact force in the zc direction (normal)
Fe_x = 0.10        # Measured contact force in the xc direction
Fe_y = 0.3         # Measured contact force in the yc direction
Fe_z = 8.00        # Measured contact force in the zc direction
Fe_tau_x = 0.1     # Measured contact torque in the xc direction
Fe_tau_y = 0.2     # Measured contact torque in the yc direction
Fe_tau_z = 0.2     # Measured contact torque in the zc direction
################################################################################


In [15]:
# Code for Task 2
# Given values
Fd = np.array([0.0, 0.0, Fd_z, 0.0, 0.0, 0.0])   # Desired task-space force/wrench
Fe  = np.array([Fe_x, Fe_y, Fe_z, Fe_tau_x, Fe_tau_y, Fe_tau_z])   # Measured task-space force/wrench

# Jacobian for the current configuration
J = np.array([
    [ 0.10,  0.20,  0.05,  0.00,  0.00,  0.00,  0.00],
    [ 0.00,  0.10,  0.20,  0.10,  0.00,  0.00,  0.00],
    [ 0.30,  0.10,  0.00,  0.20,  0.10,  0.00,  0.00],
    [ 0.00,  0.00,  0.10,  0.20,  0.30,  0.10,  0.00],
    [ 0.00,  0.10,  0.00,  0.00,  0.20,  0.30,  0.10],
    [ 0.10,  0.00,  0.00,  0.10,  0.00,  0.20,  0.30]
])      

# Compute joint-space control input for the force control
# Based on lecture notes equations (9.11) and (9.12):
# f = Fd + Kf * F_tilde        (direct force control law, eq 9.11)
# uf = J^T * Sf * f            (joint torque via Jacobian transpose, eq 9.12)

F_tilde = Fd - Fe              # Force tracking error
f = Fd + Kf * F_tilde          # Task-space force command (feedforward + feedback)
uf = J.T @ Sf @ f              # Map to joint space via Jacobian transpose

print("Joint-space force control input:")
print(uf)

Joint-space force control input:
[ 3.174e+00  1.054e+00 -3.000e-03  2.108e+00  1.039e+00 -3.300e-02
 -2.400e-02]


In [16]:
# Marking variables (don't modify)
ansT2 = uf
# End of code for Task 2